**Configs and Imports**

In [2]:
# import required libraries
import os
import numpy as np
import time
import sys
from pathlib import Path
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import re
import cv2
import numpy as np

In [3]:
#  Add project root to sys.path for local imports
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [4]:
# Import local modules
from src.configs.settings import settings
from src.utils.LPRNet import build_lprnet
from src.utils.load_data import LPRDataLoader

In [5]:
# Data Configs
INPUT_DIR = settings.LICENSE_INPUT_DIR 
TRAIN_DIR = INPUT_DIR / "train"
VAL_DIR = INPUT_DIR / "val"
OUTPUT_DIR = settings.LICENSE_OUTPUT_DIR

# Model Configs
PRETRAINED_PATH = settings.models.pretrained_lprnet
SAVE_DIR = settings.models.license_recognizer


# Training Configs
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 500
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 2e-5
T_LENGTH = 18

In [6]:
# Define the CHARS and BLANK_INDEX for LPRNet
NIGERIAN_CHARS = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
         'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K',
         'L', 'M', 'N', 'O','P', 'Q', 'R', 'S', 'T', 'U', 'V',
         'W', 'X', 'Y', 'Z',]
CTC_BLANK = "-"

# Create CHARS list and dictionary
CHARS = list(NIGERIAN_CHARS) + [CTC_BLANK]
CHARS_DICT = {char: idx for idx, char in enumerate(CHARS)}

# Calculate number of classes and blank index
NUM_CLASSES = len(CHARS)
BLANK_INDEX = len(CHARS) - 1

print("Number of classes:", NUM_CLASSES)
print("Blank index:", BLANK_INDEX)
print(CHARS)

Number of classes: 37
Blank index: 36
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '-']


**LPRNet Model**

In [7]:
# Build the LPRNet model
lpr_max_len = 10
dropout_rate = 0.5

model = build_lprnet(
    lpr_max_len=lpr_max_len,
    phase="train",
    class_num=NUM_CLASSES,
    dropout_rate=dropout_rate
)

print(model)

LPRNet(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 1, 1), padding=0, dilation=1, ceil_mode=False)
    (4): small_basic_block(
      (block): Sequential(
        (0): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
        (1): ReLU()
        (2): Conv2d(32, 32, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
        (3): ReLU()
        (4): Conv2d(32, 32, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
        (5): ReLU()
        (6): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1))
      )
    )
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool3d(kernel_size=(1, 3, 3), stride=(2, 1, 2), padding=0, dilation=1, ceil_mode=False)
    (8): small_basic_block(
      (block): Sequential(
        (0): Conv2d(64, 64, 

In [8]:
# Function to load partial pretrained weights
def load_partial_pretrained_weights(model, pretrained_path, device="cpu"):
    """
    Loads only checkpoint weights whose layer names and tensor shapes
    match the current model.

    """
    checkpoint = torch.load(pretrained_path, map_location=device)

    # Some checkpoints are saved directly as state_dict,
    # others may be wrapped inside a dictionary.
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        checkpoint = checkpoint["state_dict"]

    model_state = model.state_dict()

    loaded_layers = {}
    skipped_layers = {}

    for key, weight in checkpoint.items():
        clean_key = key.replace("module.", "")  # handles DataParallel checkpoints

        if clean_key in model_state and model_state[clean_key].shape == weight.shape:
            loaded_layers[clean_key] = weight
        else:
            skipped_layers[clean_key] = {
                "checkpoint_shape": tuple(weight.shape),
                "model_shape": tuple(model_state[clean_key].shape) if clean_key in model_state else None
            }

    model_state.update(loaded_layers)
    model.load_state_dict(model_state)

    print(f"Loaded layers: {len(loaded_layers)}")
    print(f"Skipped layers: {len(skipped_layers)}")

    return model, loaded_layers, skipped_layers

In [9]:
# Move model to device
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

# Load partial pretrained weights
model, loaded_layers, skipped_layers = load_partial_pretrained_weights(
    model=model,
    pretrained_path=PRETRAINED_PATH,
    device=device
)

Loaded layers: 54
Skipped layers: 8


In [10]:
# Print loaded and skipped layers
print("\nLoaded layers:")
for layer_name, info in skipped_layers.items():
    print(layer_name, "=>", info)


Loaded layers:
backbone.20.weight => {'checkpoint_shape': (68, 256, 13, 1), 'model_shape': (37, 256, 13, 1)}
backbone.20.bias => {'checkpoint_shape': (68,), 'model_shape': (37,)}
backbone.21.weight => {'checkpoint_shape': (68,), 'model_shape': (37,)}
backbone.21.bias => {'checkpoint_shape': (68,), 'model_shape': (37,)}
backbone.21.running_mean => {'checkpoint_shape': (68,), 'model_shape': (37,)}
backbone.21.running_var => {'checkpoint_shape': (68,), 'model_shape': (37,)}
container.0.weight => {'checkpoint_shape': (68, 516, 1, 1), 'model_shape': (37, 485, 1, 1)}
container.0.bias => {'checkpoint_shape': (68,), 'model_shape': (37,)}


**Nigerian License Plate Dataset**

In [11]:
# Verify dataset directories and list sample files
train_images = list(Path(TRAIN_DIR).glob("*"))
val_images = list(Path(VAL_DIR).glob("*"))

print("Train images:", len(train_images))
print("Val images:", len(val_images))
print("Sample train files:", train_images[:5])

Train images: 520
Val images: 129
Sample train files: [PosixPath('/home/samuel/Downloads/vision-based-access-intelligence/data/license_recognition/inputs/train/RBC254CU.jpg'), PosixPath('/home/samuel/Downloads/vision-based-access-intelligence/data/license_recognition/inputs/train/ABC216AV.jpg'), PosixPath('/home/samuel/Downloads/vision-based-access-intelligence/data/license_recognition/inputs/train/RSH500BY.jpg'), PosixPath('/home/samuel/Downloads/vision-based-access-intelligence/data/license_recognition/inputs/train/GWA410CM.jpg'), PosixPath('/home/samuel/Downloads/vision-based-access-intelligence/data/license_recognition/inputs/train/GWA19SJ.jpg')]


In [12]:
# Clean and encode labels

def clean_plate_label(label: str) -> str:
    """
    Cleans Nigerian plate labels.
    Removes spaces, hyphens, underscores, and converts to uppercase.
    """
    label = label.upper()
    label = re.sub(r"[^A-Z0-9]", "", label)
    return label


def encode_label(label: str, chars_dict: dict, max_len: int):
    """
    Converts plate string into integer character indexes.
    """
    label = clean_plate_label(label)

    if len(label) > max_len:
        raise ValueError(f"Label '{label}' is longer than max length {max_len}")

    encoded = []

    for char in label:
        if char not in chars_dict:
            raise ValueError(f"Character '{char}' not found in charset")

        encoded.append(chars_dict[char])

    return encoded, len(encoded)

In [13]:
# Test the cleaning and encoding functions 
sample_label = "LSR-456-AA"

encoded, length = encode_label(
    sample_label,
    chars_dict=CHARS_DICT,
    max_len=lpr_max_len
)

print("Original:", sample_label)
print("Cleaned:", clean_plate_label(sample_label))
print("Encoded:", encoded)
print("Length:", length)
print("Decoded:", "".join([CHARS[i] for i in encoded]))

Original: LSR-456-AA
Cleaned: LSR456AA
Encoded: [21, 28, 27, 4, 5, 6, 10, 10]
Length: 8
Decoded: LSR456AA


In [14]:
# Define the custom Dataset class for the Nigerian LPR
class NigerianLPRDataset(Dataset):
    def __init__(self, image_dir, img_size=(94, 24), max_len=10, chars_dict=None):
        """
        image_dir: folder containing cropped plate images
        img_size: (width, height)
        max_len: maximum plate label length
        chars_dict: character-to-index mapping
        """
        self.image_dir = Path(image_dir)
        self.img_size = img_size
        self.max_len = max_len
        self.chars_dict = chars_dict

        self.image_paths = []

        valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

        for path in self.image_dir.iterdir():
            if path.suffix.lower() in valid_exts:
                label = clean_plate_label(path.stem)

                if 0 < len(label) <= self.max_len:
                    self.image_paths.append(path)

        if len(self.image_paths) == 0:
            raise ValueError(f"No valid images found in {self.image_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]

        label_text = clean_plate_label(image_path.stem)
        label, length = encode_label(
            label_text,
            chars_dict=self.chars_dict,
            max_len=self.max_len
        )

        img = cv2.imread(str(image_path))

        if img is None:
            raise ValueError(f"Could not read image: {image_path}")

        img = cv2.resize(img, self.img_size)
        img = img.astype(np.float32)

        # Same normalization style used by the repo test display:
        # image was restored using img * 128 + 127.5, so training image should be roughly:
        img = (img - 127.5) / 128.0

        # BGR/RGB note:
        # cv2 loads BGR. Keep BGR for now unless your pretrained repo explicitly uses RGB.
        img = np.transpose(img, (2, 0, 1))  # HWC -> CHW

        return img, label, length

In [15]:
# Define the custom collate function for the DataLoader
def nigerian_lpr_collate_fn(batch):
    imgs = []
    labels = []
    lengths = []

    for img, label, length in batch:
        imgs.append(torch.from_numpy(img))
        labels.extend(label)
        lengths.append(length)

    labels = np.asarray(labels).flatten().astype(np.int64)

    return (
        torch.stack(imgs, 0),
        torch.from_numpy(labels),
        lengths
    )

In [16]:
# Create datasets 
train_dataset = NigerianLPRDataset(
    image_dir=TRAIN_DIR,
    img_size=(94, 24),
    max_len=lpr_max_len,
    chars_dict=CHARS_DICT
)

val_dataset = NigerianLPRDataset(
    image_dir=VAL_DIR,
    img_size=(94, 24),
    max_len=lpr_max_len,
    chars_dict=CHARS_DICT
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

Train samples: 520
Val samples: 129


In [17]:
# Test the dataset and collate function
img, label, length = train_dataset[0]

print("Image shape:", img.shape)
print("Label:", label)
print("Length:", length)
print("Decoded:", "".join([CHARS[i] for i in label]))
print("Image min/max:", img.min(), img.max())

Image shape: (3, 24, 94)
Label: [27, 11, 12, 2, 5, 4, 12, 30]
Length: 8
Decoded: RBC254CU
Image min/max: -0.98828125 0.96484375


In [18]:
# Test the data loader and collate function
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    collate_fn=nigerian_lpr_collate_fn
)

images, labels, lengths = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Flattened labels shape:", labels.shape)
print("Lengths:", lengths)
print("Total label length:", sum(lengths))

Batch image shape: torch.Size([8, 3, 24, 94])
Flattened labels shape: torch.Size([64])
Lengths: [8, 8, 8, 8, 8, 8, 8, 8]
Total label length: 64


In [19]:
# Create DataLoaders for training and validation
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=nigerian_lpr_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=nigerian_lpr_collate_fn
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 17
Val batches: 5


**Helper Functions**

In [20]:
# Function to create input and target lengths for CTC loss
def make_ctc_lengths(batch_size, target_lengths, t_length=18):
    input_lengths = torch.full(
        size=(batch_size,),
        fill_value=t_length,
        dtype=torch.long
    )

    target_lengths = torch.tensor(
        target_lengths,
        dtype=torch.long
    )

    return input_lengths, target_lengths

In [21]:
# Function for greedy decoding of CTC outputs
def greedy_decode(logits, chars, blank_index):
    """
    logits: [B, C, T]
    returns: list[str]
    """
    preds = torch.argmax(logits, dim=1)  # [B, T]
    decoded_texts = []

    for pred in preds:
        prev = None
        output_chars = []

        for idx in pred.cpu().numpy().tolist():
            if idx == blank_index:
                prev = idx
                continue

            if idx != prev:
                output_chars.append(chars[idx])

            prev = idx

        decoded_texts.append("".join(output_chars))

    return decoded_texts

In [22]:
# Function to decode flattened target labels back into strings
def decode_flattened_targets(labels, lengths, chars):
    labels = labels.cpu().numpy().tolist()

    decoded = []
    start = 0

    for length in lengths:
        target = labels[start:start + length]
        decoded.append("".join(chars[int(i)] for i in target))
        start += length

    return decoded

In [23]:
# Evaluation function for LPRNet
@torch.no_grad()
def evaluate_lprnet(model, data_loader, chars, blank_index, device):
    model.eval()

    total = 0
    exact_correct = 0

    for images, labels, lengths in data_loader:
        images = images.to(device).float()
        labels = labels.to(device).long()

        logits = model(images)

        pred_texts = greedy_decode(logits, chars, blank_index)
        true_texts = decode_flattened_targets(labels, lengths, chars)

        for pred, true in zip(pred_texts, true_texts):
            total += 1
            if pred == true:
                exact_correct += 1

    accuracy = exact_correct / total if total > 0 else 0

    return accuracy

**Model Building**

In [24]:
# Freeze backbone layers, only train charset-dependent layers 
for param in model.parameters():
    param.requires_grad = False

# Train charset-dependent layers
for param in model.backbone[20].parameters():
    param.requires_grad = True

for param in model.backbone[21].parameters():
    param.requires_grad = True

for param in model.container.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Trainable params:", trainable_params)
print("Total params:", total_params)

Trainable params: 141229
Total params: 326541


In [25]:
# Define CTC loss
ctc_loss = nn.CTCLoss(
    blank=BLANK_INDEX,
    reduction="mean",
    zero_infinity=True
)

# Define optimizer
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [26]:
model = model.to(DEVICE)

best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for images, labels, lengths in progress_bar:
        images = images.to(DEVICE).float()
        labels = labels.to(DEVICE).long()

        batch_size = images.size(0)

        input_lengths, target_lengths = make_ctc_lengths(
            batch_size=batch_size,
            target_lengths=lengths,
            t_length=T_LENGTH
        )

        input_lengths = input_lengths.to(DEVICE)
        target_lengths = target_lengths.to(DEVICE)

        logits = model(images)                  # [B, C, T]
        log_probs = logits.permute(2, 0, 1)     # [T, B, C]
        log_probs = log_probs.log_softmax(2)

        loss = ctc_loss(
            log_probs,
            labels,
            input_lengths,
            target_lengths
        )

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()),
            max_norm=5.0
        )

        optimizer.step()

        running_loss += loss.item()

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_train_loss = running_loss / len(train_loader)

    val_acc = evaluate_lprnet(
        model=model,
        data_loader=val_loader,
        chars=CHARS,
        blank_index=BLANK_INDEX,
        device=DEVICE
    )

    print(
        f"Epoch {epoch}/{EPOCHS} "
        f"| Train Loss: {avg_train_loss:.4f} "
        f"| Val Exact Match Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        save_path = SAVE_DIR 
        torch.save(model.state_dict(), save_path)

        print(f"Saved best model to: {save_path}")

Epoch 1/500: 100%|██████████| 17/17 [00:00<00:00, 27.97it/s, loss=3.6799]


Epoch 1/500 | Train Loss: 4.7133 | Val Exact Match Acc: 0.0000


Epoch 2/500: 100%|██████████| 17/17 [00:00<00:00, 106.87it/s, loss=3.5419]


Epoch 2/500 | Train Loss: 3.6309 | Val Exact Match Acc: 0.0000


Epoch 3/500: 100%|██████████| 17/17 [00:00<00:00, 103.14it/s, loss=3.3929]


...

Epoch 498/500 | Train Loss: 1.9385 | Val Exact Match Acc: 0.0000


Epoch 499/500: 100%|██████████| 17/17 [00:00<00:00, 101.24it/s, loss=1.7312]


Epoch 499/500 | Train Loss: 1.9365 | Val Exact Match Acc: 0.0000


Epoch 500/500: 100%|██████████| 17/17 [00:00<00:00, 100.96it/s, loss=2.0274]


Epoch 500/500 | Train Loss: 1.9625 | Val Exact Match Acc: 0.0078


In [27]:
@torch.no_grad()
def preview_predictions(model, data_loader, n=10):
    model.eval()

    images, labels, lengths = next(iter(data_loader))
    images = images.to(DEVICE).float()
    labels = labels.to(DEVICE).long()

    logits = model(images)

    pred_texts = greedy_decode(logits, CHARS, BLANK_INDEX)
    true_texts = decode_flattened_targets(labels, lengths, CHARS)

    for i in range(min(n, len(pred_texts))):
        print(f"TRUE: {true_texts[i]}  |  PRED: {pred_texts[i]}")

preview_predictions(model, val_loader, n=10)

TRUE: GWA12CX  |  PRED: A72Y
TRUE: GWA394DL  |  PRED: AA394L
TRUE: BDG913FR  |  PRED: B913N
TRUE: P1107DCT  |  PRED: A18Y
TRUE: KRD578HM  |  PRED: AC578HV
TRUE: EPE401GU  |  PRED: EPC1GU
TRUE: KUJ702LT  |  PRED: R72X
TRUE: KRD355GK  |  PRED: A355GA
TRUE: MMR339XA  |  PRED: RA3AA
TRUE: KWL809BD  |  PRED: R809U


In [28]:
# Load the best model for evaluation or inference
best_head_path = settings.models.license_recognizer

model.load_state_dict(torch.load(best_head_path, map_location=DEVICE))
model = model.to(DEVICE)

print("Loaded:", best_head_path)

Loaded: /home/samuel/Downloads/vision-based-access-intelligence/artifacts/models/license_recognizer/lprnet.pth


In [29]:
# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze later backbone layers
for layer_idx in [11, 12, 14, 16, 17, 20, 21]:
    for param in model.backbone[layer_idx].parameters():
        param.requires_grad = True

# Always keep final container trainable
for param in model.container.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Trainable params:", trainable_params)
print("Total params:", total_params)
print("Trainable ratio:", trainable_params / total_params)

Trainable params: 265837
Total params: 326541
Trainable ratio: 0.8140999139464876


In [30]:
FINE_TUNE_EPOCHS = 100
FINE_TUNE_LR = 1e-4

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FINE_TUNE_LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

In [31]:
best_val_acc = evaluate_lprnet(
    model=model,
    data_loader=val_loader,
    chars=CHARS,
    blank_index=BLANK_INDEX,
    device=DEVICE
)

print("Starting val accuracy:", best_val_acc)

for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Fine-tune Epoch {epoch}/{FINE_TUNE_EPOCHS}"
    )

    for images, labels, lengths in progress_bar:
        images = images.to(DEVICE).float()
        labels = labels.to(DEVICE).long()

        batch_size = images.size(0)

        input_lengths, target_lengths = make_ctc_lengths(
            batch_size=batch_size,
            target_lengths=lengths,
            t_length=T_LENGTH
        )

        input_lengths = input_lengths.to(DEVICE)
        target_lengths = target_lengths.to(DEVICE)

        logits = model(images)              # [B, C, T]
        log_probs = logits.permute(2, 0, 1) # [T, B, C]
        log_probs = log_probs.log_softmax(2)

        loss = ctc_loss(
            log_probs,
            labels,
            input_lengths,
            target_lengths
        )

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()),
            max_norm=5.0
        )

        optimizer.step()

        running_loss += loss.item()

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "lr": optimizer.param_groups[0]["lr"]
        })

    avg_train_loss = running_loss / len(train_loader)

    val_acc = evaluate_lprnet(
        model=model,
        data_loader=val_loader,
        chars=CHARS,
        blank_index=BLANK_INDEX,
        device=DEVICE
    )

    scheduler.step(val_acc)

    print(
        f"Fine-tune Epoch {epoch}/{FINE_TUNE_EPOCHS} "
        f"| Train Loss: {avg_train_loss:.4f} "
        f"| Val Exact Match Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        save_path = SAVE_DIR / "best_lprnet_nigerian_finetuned.pth"
        torch.save(model.state_dict(), save_path)

        print(f"Saved best fine-tuned model to: {save_path}")

Starting val accuracy: 0.007751937984496124


Fine-tune Epoch 1/100: 100%|██████████| 17/17 [00:00<00:00, 66.76it/s, loss=2.0671, lr=0.0001]


Fine-tune Epoch 1/100 | Train Loss: 2.0913 | Val Exact Match Acc: 0.0000


Fine-tune Epoch 2/100: 100%|██████████| 17/17 [00:00<00:00, 85.12it/s, loss=2.0039, lr=0.0001]


Fine-tune Epoch 2/100 | Train Loss: 2.0643 | Val Exact Match Acc: 0.0000


...

Fine-tune Epoch 98/100 | Train Loss: 1.9871 | Val Exact Match Acc: 0.0000


Fine-tune Epoch 99/100: 100%|██████████| 17/17 [00:00<00:00, 81.54it/s, loss=1.9459, lr=1.22e-8]


Fine-tune Epoch 99/100 | Train Loss: 1.9799 | Val Exact Match Acc: 0.0000


Fine-tune Epoch 100/100: 100%|██████████| 17/17 [00:00<00:00, 82.80it/s, loss=1.8649, lr=1.22e-8]


Fine-tune Epoch 100/100 | Train Loss: 1.9774 | Val Exact Match Acc: 0.0000


In [32]:
best_finetuned_path = SAVE_DIR / "best_lprnet_nigerian_finetuned.pth"

model.load_state_dict(torch.load(best_finetuned_path, map_location=DEVICE))
model = model.to(DEVICE)

preview_predictions(model, val_loader, n=20)

NotADirectoryError: [Errno 20] Not a directory: '/home/samuel/Downloads/vision-based-access-intelligence/artifacts/models/license_recognizer/lprnet.pth/best_lprnet_nigerian_finetuned.pth'